 **This Notebook contains script for Loading cleaned parquet Dataframes , Cardindality analysis of all columns from all Tables** 
**Framing Machine learning Problem and Preparing Dataset for Machine learning Problem (by including only orders that are delivered as for our Machine learning Problem that is only relevant in training  ) ,Merging Datasets into final  Dataset by Joinig all Tables , Granularity of Final Merged Table -order_item level and we did Basic level EDA and target variable Analysis**

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

from pathlib import Path

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
DATA_DIR = PROJECT_ROOT / "data"
ARTIFACT_DIR = PROJECT_ROOT / "artifact"
ARTIFACT_DIR.mkdir(exist_ok=True)

path = ARTIFACT_DIR
# sns.set_theme(style='whitegrid', palette='muted')

# Load cleaned tables
customers    = pd.read_parquet(f'{path}/customers_clean.parquet')
sellers      = pd.read_parquet(f'{path}/sellers_clean.parquet')
products     = pd.read_parquet(f'{path}/products_clean.parquet')
orders       = pd.read_parquet(f'{path}/orders_clean.parquet')
order_items  = pd.read_parquet(f'{path}/order_items_clean.parquet')
payments     = pd.read_parquet(f'{path}/payments_clean.parquet')
reviews      = pd.read_parquet(f'{path}/reviews_clean.parquet')
geolocations = pd.read_parquet(f'{path}/geolocations_clean.parquet')

print("All parquets loaded ✓")


In [ ]:
## Column type classification 

# ─── Helper function tp : classify a single column ────────────────────────────────────────
def classify_column(series):
    nuniq = series.nunique()
    dtype = str(series.dtype)

    if 'datetime' in dtype:
        return 'Datetime'
    if dtype in ('string','object'):
        if nuniq <= 15:
            return 'Categorical Nominal (Low)'
        elif nuniq <= 100:
            return 'Categorical Nominal (Medium)'
        else:
            return 'High Cardinality'
    if dtype in ('bool',):
        return 'Categorical Nominal (Low)'
    # numeric
    if nuniq <= 15:
        return 'Numerical Discrete'
    if series.dropna().apply(float.is_integer).all() if dtype != 'float32' else False:
        return 'Numerical Discrete'
    return 'Numerical Continuous'

dfs = {
    'customers': customers, 'sellers': sellers, 'products': products,
    'orders': orders, 'order_items': order_items, 'payments': payments,
    'reviews': reviews, 'geolocations': geolocations
}

rows = []
for tbl, df in dfs.items():
    for col in df.columns:
        rows.append({
            'table'        : tbl,
            'column'       : col,
            'dtype'        : str(df[col].dtype),
            'unique_values': df[col].nunique(),
            'category_type': classify_column(df[col])
        })
## Summary of each tables columns
col_summary = pd.DataFrame(rows)
print(col_summary.to_string(index=False))


# Cardinality analysis 

Find columns with:

low cardinality\
medium cardinality\
high cardinality

Questions:

Which columns are dangerous for One-Hot Encoding?\
Which columns should use frequency encoding?\
Which columns are ordinal?




In [ ]:
# ## CARDINALITY SUMMARY OF EACH COLUMN 

# cat_cols_all = col_summary[col_summary['category_type'].str.startswith('Cat') | 
#                                (col_summary['category_type']=='High Cardinality')]

# print("=" * 65)
# print(f"{'Column':<40} {'Unique':>8}  Category")
# print("=" * 65)
# for _, row in cat_cols_all.sort_values('unique_values', ascending=False).iterrows():
#     print(f"{row['table']}.{row['column']:<35} {row['unique_values']:>8}  {row['category_type']}")

# encoding_notes = (
#     "ENCODING DECISIONS\n\n"
    
#  " Dangerous for One-Hot  : geolocation_city (8011), customer_city (4119),\n"
    
#     "                     seller_city (611), product_category_name (73)\n\n"
    
#  " Should use Freq/Target : city columns, zip codes\n\n"
    
#  " Ordinal                : review_score (1-5)]n]n"
    
#  " Safe for One-Hot       : payment_type (5), order_status (8),]n"
    
#           "             customer_state (27), seller_state (23)"
# )
# print(encoding_notes)


In [ ]:
## Manual CARDINALITY ANALYSIS 

## categorical columns 
lt=[]
for name,df in dfs.items():
    lt.extend(df.select_dtypes(include=['category','category','string']).columns.to_list())
    
print("Catgorical columns are : ")
print(lt)

## Categorical columns :

# customers -(Entity )
# customer_city dtype
# customer_state dtype 

# sellers (Entity)

# seller_city
# seller_state 

# products (Entity)

# product_category_name 

# orders (Entity)

# order_status 
# order_purchase_timestamp

# payments (Entity)
# payment_type

# reviews (entity)
# review_comment_title 
# review_comment_message 

# geolocations (entity)
# geolocation_city
# geolocation_state



print("No of different cities we have customers in :",customers['customer_city'].nunique())
print(customers['customer_state'].unique())
print("number of unique states in which we have customers ",customers['customer_state'].nunique())
print("Number od different cities we have sellers ",sellers['seller_city'].nunique())
print("Number of unique states in which we have sellers ",sellers['seller_state'].unique())

# print("Different products categories we have ",products['product_category_name'].unique())
print("Number of different Product categories we have ", products['product_category_name'].nunique())

print("Different order status ",orders['order_status'].unique())

print("No of different types of order status ",orders['order_status'].nunique())
print("Different types of payments type ",payments['payment_type'].unique())
print("Number of different types of payments type ",payments['payment_type'].nunique())

print("geoloction cities ",geolocations['geolocation_city'].nunique())
print("geolpcations state ",geolocations['geolocation_state'].nunique())

# print("Unique product items   ",order_items['product_id'].nunique())
print(orders['order_id'].nunique())
print(reviews['review_id'].nunique())
print("Number of different types of reviews ",reviews['review_comment_title'].nunique())

## Cardinality Report 

**Largest cardinality columns** (based on  output):
- `geolocation_city` – **8,011 unique values**
- `customer_city` – **4,119**
- `seller_city` – **611**
- `product_category_name` – **73** (moderate)

**Lowest cardinality columns**:
- `payment_type` – **5**
- `order_status` – **8**
- `review_score` – **5** (1–5)
- `customer_state` – **27**
- `seller_state` – **23**

---


## Machine Learning problem 

Delivery Delay prediction 
```python Possible targets :
    Regression
    delivery_delay_days 
    
    Classification 
    is_delayed
```

| Dataset | Why Needed |
|---|---|
| orders_clean | purchase & delivery dates |
| order_items_clean | freight/product/seller info |
| customers_clean | customer location |
| sellers_clean | seller location |
| products_clean | product category |
| payments_clean | payment behavior |
| geolocation_clean | optional distance features |


In [ ]:
## Dataset Preparation for Machine Learning 

for col in orders.columns.to_list():
    print(f"{col} ",orders[col].dtype)


orders['order_purchase_timestamp'] = pd.to_datetime(
    orders['order_purchase_timestamp']
)



## In our final prediction dataset we have to include only orders that are delivered 

orders_ml=orders[orders['order_status']=='delivered'].copy()

## Remove orders for which delivery dates are missing

orders_ml=orders_ml.dropna(subset=['order_delivered_customer_date','order_estimated_delivery_date'])

# Regression Target create 

orders_ml['order_delivery_delay']=(orders_ml['order_delivered_customer_date']-
                                      orders_ml['order_estimated_delivery_date']
                                      ).dt.days

## Classification Target create 

orders_ml['is_delayed']=(
    orders_ml['order_delivery_delay']>0
).astype('int')






In [ ]:
## Target Variable analysis 

print("Regression Target ,order delivery delay days \n",orders_ml['order_delivery_delay'].describe().round(2))

print("Classification target ,order delayed or not ?: \n",orders_ml['is_delayed'].describe().round(2))


In [ ]:
# Visualization Target Variables 

import matplotlib.pyplot as plt
import seaborn as sns
fig, axes = plt.subplots(1, 3, figsize=(16, 4))


## Regression Target 

## order_delivery_delay_distribution -clipped (for Better visibilty)-Histogram plot 

clipped=orders_ml['order_delivery_delay'].clip(-30,60)

# How clipping works?



sns.histplot(clipped,kde=True,ax=axes[0],color='steelblue')
axes[0].axvline(0,color='red',linestyle='--',label='On time Delivery ')
axes[0].set_title('Delivery Delay Distribution (days)')
axes[0].set_xlabel('Delay days (negative = early)')
axes[0].legend()
## Order delivery delay Distribution -box plot 
sns.boxplot(x=orders_ml['order_delivery_delay'].clip(-30, 60), ax=axes[1], color='lightcoral')
axes[1].axvline(0, color='red', linestyle='--')
axes[1].set_title('Delay Boxplot')
plt.tight_layout()

## Classification target 

## count plot - for class imbalance checking

class_counts=orders_ml['is_delayed'].value_counts()
axes[2].pie(class_counts,labels=['On time ','Delayed'],colors=['#4CAF50','#F44336'],autopct='%1.1f%%',startangle=90)
axes[2].set_title("Class Imbalance ")


plt.savefig(ARTIFACT_DIR / "target_distribution.png", dpi=200, bbox_inches='tight')
plt.tight_layout()
plt.show()

In [ ]:
# Review score distribution

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

reviews['review_score'] = pd.to_numeric(reviews['review_score'], errors='coerce')
sns.countplot(x=reviews['review_score'].dropna(), ax=axes[0], palette='RdYlGn')
axes[0].set_title('Review Score Distribution')

# Payment type
payments['payment_type'] = payments['payment_type'].astype(str)
pay_counts = payments['payment_type'].value_counts()
sns.barplot(x=pay_counts.index, y=pay_counts.values, ax=axes[1], palette='Blues_r')
axes[1].set_title('Payment Type Distribution')
axes[1].set_xlabel('Payment Type')
axes[1].set_ylabel('Count')

plt.tight_layout()
plt.savefig(f'{path}/eda_review_payment.png', dpi=150, bbox_inches='tight')
plt.show()


In [ ]:
# ==============================
# IQR-Based Extreme Delay Flags
# Regression Pipeline
# ==============================

# Calculate Q1 and Q3
Q1 = orders_ml['order_delivery_delay'].quantile(0.25)
Q3 = orders_ml['order_delivery_delay'].quantile(0.75)

# IQR
IQR = Q3 - Q1

# Bounds
lower_bound = Q1 - 1.5 * IQR
upper_bound = Q3 + 1.5 * IQR

print(f"Q1           : {Q1}")
print(f"Q3           : {Q3}")
print(f"IQR          : {IQR}")
print(f"Lower Bound  : {lower_bound}")
print(f"Upper Bound  : {upper_bound}")

# ==========================================
# Create Outlier Flags (DO NOT DROP YET)
# ==========================================

# Lower extreme flag
orders_ml['extreme_early_flag'] = (
    orders_ml['order_delivery_delay'] < lower_bound
).astype(int)

# Upper extreme flag
orders_ml['extreme_late_flag'] = (
    orders_ml['order_delivery_delay'] > upper_bound
).astype(int)

# Combined flag

orders_ml['delay_outlier_flag'] = (
    (orders_ml['order_delivery_delay'] < lower_bound) |
    (orders_ml['order_delivery_delay'] > upper_bound)
).astype(int)

# ==========================================
# Summary
# ==========================================

print("\nOutlier Summary:")
print(orders_ml['delay_outlier_flag'].value_counts())

print("\nExtreme Early Orders:")
print(orders_ml['extreme_early_flag'].sum())

print("\nExtreme Late Orders:")
print(orders_ml['extreme_late_flag'].sum())

In [ ]:
orders_ml.columns

In [ ]:

## Merge Datasets 


## Merge order_items

ml_df=orders_ml.merge(order_items,on='order_id',how='left')

## This is purposely Joined using left Join since we join all other table from base table using 
# left Join , so no dataset is lost 

# merge customers 

ml_df=ml_df.merge(customers ,on='customer_id',how='left')

# merge sellers 
ml_df=ml_df.merge(sellers,on='seller_id',how='left')


# merge products 

ml_df=ml_df.merge(products,on='product_id',how='left')

# Merge Payments 

ml_df=ml_df.merge(payments,on='order_id',how='left')

# Merge reviews

ml_df=ml_df.merge(reviews,on='order_id',how='inner')


In [ ]:
ml_df.columns


In [ ]:
base_ml_df=ml_df.copy()
base_ml_df.columns


In [ ]:
file_path = ARTIFACT_DIR / "ml_df.parquet"
ml_df.to_parquet(file_path,engine="pyarrow",compression="snappy",index=False)
print("saved successfully")